# Челленджи недели: классы, наследование, магические методы, dataclass

**Цель:** проверить, что концепты ООП работают вместе и встраиваются в инфраструктуру Python (`len`, `==`, `for`, `dict`-ключи, `print`).

**Как работать:**
- Часть задач сопровождается ячейкой «Объясни своими словами» — там нужно не только написать код, но и сформулировать, почему он работает.
- Задачи решаются на 5-25 строк кода. На W3 разрешены и `def`, и `class` — выбираем по задаче.
- Сначала попробуй сам, без подглядываний. Если застрял на 10+ минут — открой solution-версию.


## Задание 1: Класс `Vector` с арифметикой и хэшем

Реализуй класс `Vector` для двумерного вектора:

- `__init__(self, x, y)` — два координаты
- `__repr__` — формат `Vector(x=3, y=4)` (валидный Python-код)
- `__eq__` — два вектора равны, если совпадают `x` и `y`; для чужих типов верни `NotImplemented`
- `__hash__` — через `hash((self.x, self.y))`
- `__add__` — складывает два вектора покомпонентно, возвращает новый `Vector`

Проверка: `Vector(1, 2) + Vector(3, 4) == Vector(4, 6)` должно быть `True`, и оба объекта должны корректно работать как ключи `dict` или элементы `set`.

In [1]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector(x={self.x}, y={self.y})"

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)                    # Vector(x=4, y=6)
print(v1 + v2 == Vector(4, 6))    # True
print({Vector(0, 0), Vector(0, 0), Vector(1, 1)})  # set из 2 элементов


Vector(x=4, y=6)
True
{Vector(x=1, y=1), Vector(x=0, y=0)}


**Объясни своими словами:** почему нужно реализовывать `__hash__` вместе с `__eq__`, и что произойдёт, если оставить только `__eq__`?

Когда вы определяете `__eq__`, Python автоматически делает класс **нехэшируемым** (`__hash__` устанавливается в `None`). Это сделано из соображений корректности: если объект попал в `set` или как ключ в `dict`, его хэш используется для быстрого поиска по бакетам. Контракт: равные объекты обязаны иметь равные хэши. Если кастомное `__eq__` сравнивает по содержимому, а `__hash__` остаётся унаследованный (по адресу в памяти), правило ломается — два равных вектора окажутся в разных бакетах. Python предпочитает упасть с `TypeError: unhashable type` сразу, чем тихо хранить дубликаты в `set`. Решение — определить `__hash__` через те же атрибуты, что и `__eq__`: `hash((self.x, self.y))`.

## Задание 2: `@dataclass(frozen=True)` как ключ словаря

Перепиши класс `Point` (двумерная точка) через `@dataclass(frozen=True)`. Поля — `x: int`, `y: int`.

Затем используй точки как **ключи словаря**, чтобы посчитать, сколько раз каждая точка встретилась в списке `points`.

Подсказка: `frozen=True` автоматически генерирует `__hash__` — никаких дополнительных методов писать не нужно. Сравни с заданием 1 по объёму кода.

In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: int
    y: int

points = [
    Point(0, 0), Point(1, 2), Point(0, 0),
    Point(3, 4), Point(1, 2), Point(0, 0),
]

counts: dict[Point, int] = {}
for p in points:
    counts[p] = counts.get(p, 0) + 1

for point, n in counts.items():
    print(point, "→", n)
# Point(x=0, y=0) → 3
# Point(x=1, y=2) → 2
# Point(x=3, y=4) → 1


Point(x=0, y=0) → 3
Point(x=1, y=2) → 2
Point(x=3, y=4) → 1


## Задание 3: Счётчик созданных экземпляров через атрибут класса

Реализуй класс `Widget`, который считает, сколько всего экземпляров было создано за время работы программы. Используй **атрибут класса** `instances_created` как счётчик: при каждом вызове `__init__` увеличивай его на 1.

Класс хранит имя в `self.name`. Метод-классметод `total()` возвращает текущее значение счётчика.

Проверка:
- создай 3 виджета
- `Widget.total()` должно вернуть `3`
- `Widget.instances_created` тоже `3`

In [3]:
class Widget:
    instances_created = 0   # атрибут класса — общий счётчик

    def __init__(self, name):
        self.name = name
        Widget.instances_created += 1  # обращаемся через имя класса

    @classmethod
    def total(cls):
        return cls.instances_created

a = Widget("button")
b = Widget("slider")
c = Widget("checkbox")

print(Widget.total())               # 3
print(Widget.instances_created)     # 3
print(a.name, b.name, c.name)       # button slider checkbox


3
3
button slider checkbox


**Объясни своими словами:** почему мы пишем `Widget.instances_created += 1`, а не `self.instances_created += 1`? Что произойдёт во втором варианте?

`self.instances_created += 1` Python разворачивает в `self.instances_created = self.instances_created + 1`. На правой стороне Python ищет `instances_created` через MRO — у экземпляра нет, поднимается к классу, находит `0`. На левой стороне — присваивание через `self`, то есть **создание атрибута экземпляра** с тем же именем. После первого вызова у объекта появляется свой `instances_created = 1`, который *перекрывает* атрибут класса для этого объекта. Атрибут класса остаётся `0`. Каждый следующий вызов `__init__` повторит то же самое — каждый объект получит свою копию со значением `1`. Счётчик не работает. Чтобы менять именно атрибут класса, обращаемся через имя класса: `Widget.instances_created += 1` — здесь и чтение, и запись идут к классу.

## Задание 4: Класс `Stack` со sequence-протоколом

Реализуй класс `Stack` (стек, LIFO):

- `__init__()` — пустой стек
- `push(item)` — кладёт элемент наверх
- `pop()` — снимает и возвращает верхний элемент (если пусто — `IndexError`)
- `__len__` — текущая высота стека
- `__getitem__(i)` — индексация, `0` это нижний элемент
- `__repr__` — формат `Stack([1, 2, 3])`

Проверка: благодаря паре `__len__` + `__getitem__` стек должен **автоматически** поддерживать `for`-цикл и `list()` без `__iter__` — это и есть sequence-протокол.

In [4]:
class Stack:
    def __init__(self):
        self._items: list = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def __len__(self):
        return len(self._items)

    def __getitem__(self, index):
        return self._items[index]

    def __repr__(self):
        return f"Stack({self._items!r})"

s = Stack()
s.push("first")
s.push("second")
s.push("third")

print(s)                # Stack(['first', 'second', 'third'])
print(len(s))           # 3
print(s[0])             # first — нижний
print(s[-1])            # third — верхний

# for-цикл и list() работают автоматически благодаря sequence-протоколу
for item in s:
    print("  iter:", item)

print(list(s))          # ['first', 'second', 'third']
print(s.pop())          # third
print(len(s))           # 2


Stack(['first', 'second', 'third'])
3
first
third
  iter: first
  iter: second
  iter: third
['first', 'second', 'third']
third
2


**Объясни своими словами:** почему `for x in stack:` работает без явного `__iter__`? Что Python делает за нас?

Python хранит два протокола итерации. **Iterator-протокол** — `__iter__` + `__next__`, явная итерация. **Sequence-протокол** — `__len__` + `__getitem__` с целочисленным индексом от `0`. Если у объекта нет `__iter__`, Python пытается итерировать через sequence-протокол: вызывает `obj[0]`, `obj[1]`, `obj[2]`, … пока не получит `IndexError`. Это автоматический fallback, ради которого достаточно реализовать `__getitem__`. Те же `for`, `list()`, `tuple()`, `random.choice` начинают работать без дополнительного кода. У встроенных типов (`list`, `tuple`, `str`) есть оба протокола; у нашего `Stack` — только sequence, и его нам достаточно.

## Задание 5: Иерархия `Animal` → `Dog` / `Cat` с полиморфизмом

Реализуй три класса:

- `Animal` — `__init__(self, name)`, метод `speak(self)` возвращает строку `"..."` (заглушка)
- `Dog(Animal)` — переопределяет `speak`, возвращает `"гав"`
- `Cat(Animal)` — переопределяет `speak`, возвращает `"мяу"`

Затем напиши **функцию** `chorus(animals)`, которая принимает список животных и возвращает строку вида `"Барсик: мяу; Рекс: гав; Мурка: мяу"`. Функция не должна знать конкретный тип — просто вызывает `animal.speak()`. Это и есть полиморфизм.

In [5]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self):
        return "гав"

class Cat(Animal):
    def speak(self):
        return "мяу"

def chorus(animals):
    parts = [f"{a.name}: {a.speak()}" for a in animals]
    return "; ".join(parts)

animals = [Cat("Барсик"), Dog("Рекс"), Cat("Мурка")]
print(chorus(animals))
# Барсик: мяу; Рекс: гав; Мурка: мяу


Барсик: мяу; Рекс: гав; Мурка: мяу


## Задание 6: `@classmethod` фабрика — парсинг CSV-строки

Реализуй `@dataclass` `Employee` с полями `name: str`, `salary: int`, `role: str`. Добавь к нему `@classmethod from_csv_row(cls, row)`, который принимает строку формата `"Аня,100000,engineer"` и возвращает новый объект `Employee`.

Подсказка: используй `cls(...)`, а не `Employee(...)` — это правильный паттерн фабрики, который корректно работает при наследовании.

Затем примени фабрику к списку строк через `map`/list-comprehension.

In [6]:
from dataclasses import dataclass

@dataclass
class Employee:
    name: str
    salary: int
    role: str

    @classmethod
    def from_csv_row(cls, row):
        name, salary, role = row.split(",")
        return cls(name=name.strip(), salary=int(salary), role=role.strip())

rows = [
    "Аня,100000,engineer",
    "Боря,120000,manager",
    "Вера,90000,analyst",
]

employees = [Employee.from_csv_row(r) for r in rows]
for emp in employees:
    print(emp)
# Employee(name='Аня', salary=100000, role='engineer')
# Employee(name='Боря', salary=120000, role='manager')
# Employee(name='Вера', salary=90000, role='analyst')


Employee(name='Аня', salary=100000, role='engineer')
Employee(name='Боря', salary=120000, role='manager')
Employee(name='Вера', salary=90000, role='analyst')


**Объясни своими словами:** что произойдёт, если внутри `from_csv_row` написать `Employee(...)` вместо `cls(...)` и потом унаследоваться от `Employee`?

Если в фабрике написать жёстко `Employee(...)`, то любой наследник унаследует метод как есть, и при вызове `Manager.from_csv_row(...)` метод будет создавать **`Employee`**, а не `Manager`. То есть наследник теряет смысл фабрики — он не может через тот же интерфейс получить объект своего типа. С `cls(...)` Python подставит фактический класс, через который вызвали метод: `Manager.from_csv_row(...)` создаст `Manager`, `Employee.from_csv_row(...)` — `Employee`. Это та же причина, по которой `from_pretrained` в HuggingFace объявлен один раз в базовом классе, но возвращает `BertModel`, `GPT2Model`, etc. — в зависимости от того, через какой класс его вызвали.

## Задание 7: `@property` с валидацией в сеттере

Реализуй класс `Temperature` с одним полем — температурой в Цельсиях. Используй `@property` для геттера и `@<имя>.setter` для сеттера, чтобы при попытке поставить значение ниже `-273.15` (абсолютный ноль) — поднимать `ValueError`.

Подсказки:
- внутреннее значение храни в `self._celsius` (соглашение: подчёркивание = «приватный»)
- геттер: `@property def celsius(self): return self._celsius`
- сеттер: `@celsius.setter def celsius(self, value): ...` с проверкой
- в `__init__` тоже используй сеттер через `self.celsius = ...`, чтобы валидация сработала и при создании

Проверка: `Temperature(-300)` должно упасть с `ValueError`; `t.celsius = 25` — работать.

In [7]:
class Temperature:
    def __init__(self, celsius):
        # используем сеттер, чтобы валидация сработала и при создании
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(
                f"Температура {value}°C ниже абсолютного нуля (-273.15°C)"
            )
        self._celsius = value

t = Temperature(20)
print(t.celsius)        # 20

t.celsius = 25          # работает через сеттер
print(t.celsius)        # 25

# Валидация при создании
try:
    bad = Temperature(-300)
except ValueError as e:
    print("поймали:", e)

# Валидация при присваивании
try:
    t.celsius = -500
except ValueError as e:
    print("поймали:", e)


20
25
поймали: Температура -300°C ниже абсолютного нуля (-273.15°C)
поймали: Температура -500°C ниже абсолютного нуля (-273.15°C)


# Готово

Ты только что прошёл задачи на пересечении классов, наследования, магических методов, `@dataclass` и `@classmethod`. Если все ячейки прошли — концепты ООП у тебя работают как единое целое.

На следующей неделе мы разберём память Python, потоки и асинхронность — и увидим, как классы из этой недели становятся базой для async-контекстных менеджеров (`__aenter__`/`__aexit__`) и собственных типов в многопоточных программах.
